# End-Sem Laboratory Project: Dynamic Hybrid ML System for Anomaly Detection and Classification of Health Signals in WBANs

**Author:** Harshal Shah (2023A7PS0055H)
**Guide:** Prof. Chittaranjan Hota — BITS Pilani, Hyderabad Campus
**Course:** CS F366

This notebook implements the full **Dynamic Two-Stage Hybrid ML Framework** described in the project report:

- **Stage 1 — Gatekeeper Classification:** three interchangeable models (SVM, Random Forest, Bi-Directional LSTM) screen each observation as Normal / Abnormal.
- **Stage 2 — SVR-Based Triage:** flagged observations are categorised into `01` (Contextual Anomaly), `10` (Point Anomaly), or `11` (Sensor Fault) using patient-adaptive dynamic thresholds and SVR residual scoring.

**Changes from the previous version of this notebook:**
1. Added the **Random Forest** and **Bi-Directional LSTM** gatekeepers (previously only SVM–SVR was implemented, despite the report describing all three).
2. Fixed the **dynamic bounds calculation** (Stage 2) to use the mean/std of the *normal-only* training subset, matching Eq. 6–9 in the report exactly (the previous version used the full training set's standard deviation).
3. Refactored the script into reusable functions with a **unified `run_inference()` interface** (Section 3.5 / Listing 1 of the report), so the same Stage 2 logic serves all three gatekeepers — and so this logic can be shared with `wban_dashboard.py`.
4. Added a **Colab/local fallback** for data loading — this notebook still uses the Colab upload widget when run in Colab, but now also works in a plain local Jupyter kernel by reading CSVs from `./data`.
5. Added diagnostics that surface **class-imbalance / missing-class warnings** (the previous run showed zero test-set support for the `01` and `11` classes — this is now flagged explicitly rather than silently producing a misleading macro F1).
6. Generates the full comparison table, confusion matrices, BiLSTM training curves, and Random Forest feature importances for **all three models**, matching the report's Section 5 figures — instead of just one model's results.

> **Note on the "Static Threshold" baseline in the report's Table 5:** that baseline is not implemented here, since fabricating a number without a corresponding classifier would misrepresent the results. See the note in Phase 7b if you want to add one.


In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================
import io
import os
import re
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
)
from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

# ---- Global configuration (mirrors Section 3 of the project report) ----
FEATURES = ["Body_Temperature", "Heart_Rate", "Pulse_Rate", "SpO2", "ECG"]
ROLL_WINDOW = 5          # rolling window length, w (Eq. 1)
LOOKBACK = 5              # BiLSTM sequence lookback, L (Eq. 5)
TEST_SIZE = 0.20
RANDOM_STATE = 42

CLASS_NAMES = {
    "00": "Normal (00)",
    "01": "Contextual Anomaly (01)",
    "10": "Point Anomaly (10)",
    "11": "Sensor Fault (11)",
}

print("[Setup] Imports complete.")

In [ ]:
# ==========================================================
# CORE PIPELINE FUNCTIONS — Data Ingestion
# ==========================================================
def numerical_sort_key(value):
    """Sorts filenames like '2.csv' before '10.csv' (natural/numeric order)."""
    parts = re.split(r"(\d+)", value)
    parts[1::2] = map(int, parts[1::2])
    return parts


def load_uploaded_files():
    """
    Loads telemetry CSVs.

    Uses the Colab upload widget when running in Google Colab (matching the
    original workflow), and transparently falls back to reading every CSV
    found under ./data when run in a standard local Jupyter kernel — so this
    notebook no longer hard-depends on Colab to execute.
    """
    try:
        from google.colab import files  # noqa: F401

        print("Please upload the patient telemetry CSV files.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No files uploaded.")
        return dict(uploaded.items())
    except ImportError:
        print("[Info] Not running in Colab — loading CSVs from ./data instead.")
        paths = sorted(glob.glob(os.path.join("data", "*.csv")), key=numerical_sort_key)
        if not paths:
            raise FileNotFoundError(
                "No CSV files found in ./data. Place telemetry CSVs there, "
                "or run this notebook in Google Colab to use the upload widget."
            )
        return {os.path.basename(p): open(p, "rb").read() for p in paths}


def aggregate_telemetry(raw_files):
    """Concatenates uploaded/local CSVs in numerical filename order and
    derives the four-code ground-truth label (Section 3.2.2 of the report)."""
    sorted_names = sorted(raw_files.keys(), key=numerical_sort_key)
    frames = [pd.read_csv(io.BytesIO(raw_files[name])) for name in sorted_names]
    df = pd.concat(frames, ignore_index=True)

    try:
        df["Actual_Code"] = df.iloc[:, 5].astype(str) + df.iloc[:, 6].astype(str)
    except IndexError:
        raise ValueError(
            "Expected the Anomaly_Type and Normal columns at positions 5 and 6 "
            "of each CSV — check the uploaded file schema."
        )

    return df, sorted_names

In [ ]:
# ==========================================================
# CORE PIPELINE FUNCTIONS — Feature Engineering & Scaling
# ==========================================================
def engineer_features(df):
    """Adds rolling mean/std features per sensor channel (Eq. 1) and
    backward/forward-fills the NaNs the rolling window introduces."""
    df = df.copy()
    for col in FEATURES:
        df[f"{col}_roll_mean"] = df[col].rolling(window=ROLL_WINDOW).mean()
        df[f"{col}_roll_std"] = df[col].rolling(window=ROLL_WINDOW).std()
    df = df.bfill().ffill()

    X = df.drop(columns=["Anomaly_Type", "Normal", "Actual_Code"])
    y = df["Actual_Code"].astype(str)
    return df, X, y


def split_and_scale(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """Stratified train/test split followed by Min-Max scaling fit only on
    the training partition (Eq. 2), preventing test-set leakage."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = MinMaxScaler()
    scaler.fit(X_train)

    X_train_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=X_train.columns, index=X_train.index
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test), columns=X_test.columns, index=X_test.index
    )

    # Reset indices so every returned frame/series lines up positionally.
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    X_train_scaled = X_train_scaled.reset_index(drop=True)
    X_test_scaled = X_test_scaled.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    return X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test, scaler

In [ ]:
# ==========================================================
# CORE PIPELINE FUNCTIONS — Stage 1: Gatekeeper Models
# ==========================================================
def train_svm_gatekeeper(X_train_scaled, y_train):
    """Binary SVM gatekeeper: Normal (0) vs Abnormal (1). Section 3.3.1."""
    y_binary = (y_train != "00").astype(int)
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_res, y_res = ros.fit_resample(X_train_scaled, y_binary)

    svm = SVC(kernel="rbf", C=10.0, gamma="scale", random_state=RANDOM_STATE)
    svm.fit(X_res, y_res)
    return svm


def train_rf_gatekeeper(X_train_scaled, y_train):
    """Multiclass Random Forest gatekeeper predicting the anomaly code
    directly, with balanced class weighting. Section 3.3.2."""
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_train)

    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_res, y_res = ros.fit_resample(X_train_scaled, y_encoded)

    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_res, y_res)
    return rf, le


def build_windowed_input(X_scaled_df, lookback=LOOKBACK):
    """
    Builds a windowed 3-D tensor of shape (n, lookback, d), aligned 1:1 with
    the rows of X_scaled_df. The first `lookback - 1` rows are padded by
    repeating the first available row, so every row in the input receives a
    prediction (this differs slightly from Eq. 5's raw (n-L, L, d) shape,
    which would drop the first L rows — padding keeps gatekeeper output
    lengths consistent across SVM / RF / BiLSTM for a fair comparison).
    """
    X_arr = np.asarray(X_scaled_df)
    n = len(X_arr)
    d = X_arr.shape[1] if n else 0
    if n == 0:
        return np.empty((0, lookback, d))

    pad = np.repeat(X_arr[[0]], max(lookback - 1, 0), axis=0)
    padded = np.vstack([pad, X_arr])
    return np.array([padded[i : i + lookback] for i in range(n)])


def train_bilstm_gatekeeper(X_train_scaled, y_train, epochs=50, patience=5, verbose=0):
    """Bi-Directional LSTM gatekeeper over 5-step windows. Section 3.3.3 /
    Table 3. Requires TensorFlow; raises a clear error if unavailable."""
    try:
        from tensorflow.keras.callbacks import EarlyStopping
        from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Input, LSTM
        from tensorflow.keras.models import Sequential
    except ImportError as exc:
        raise ImportError(
            "TensorFlow is required for the BiLSTM gatekeeper. "
            "Install it with `pip install tensorflow` or deselect BiLSTM."
        ) from exc

    le = LabelEncoder()
    y_encoded = le.fit_transform(y_train)

    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_res, y_res = ros.fit_resample(X_train_scaled, y_encoded)
    X_res = pd.DataFrame(X_res, columns=X_train_scaled.columns)

    X_win = build_windowed_input(X_res, lookback=LOOKBACK)
    n_classes = len(le.classes_)

    model = Sequential(
        [
            Input(shape=(LOOKBACK, X_res.shape[1])),
            Bidirectional(LSTM(64)),
            Dropout(0.2),
            Dense(n_classes, activation="softmax"),
        ]
    )
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    early_stop = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)
    history = model.fit(
        X_win,
        y_res,
        validation_split=0.15,
        epochs=epochs,
        batch_size=64,
        callbacks=[early_stop],
        verbose=verbose,
    )
    return model, le, history

In [ ]:
# ==========================================================
# CORE PIPELINE FUNCTIONS — Stage 2: SVR Triage & Dynamic Thresholds
# ==========================================================
def fit_dynamic_bounds(X_train_orig, y_train):
    """
    Patient-adaptive dynamic bounds, computed exclusively from the NORMAL
    training subset (Section 3.4.1, Eq. 6-9). Uses the mean/std of the
    normal-only rows, matching the report's equations exactly — this fixes
    a discrepancy in the original notebook, which used the full training
    set's standard deviation instead of the normal-subset statistics.
    """
    normal_mask = (y_train == "00").to_numpy()
    X_normal = X_train_orig[normal_mask]

    bounds, min_jumps, fault_thresholds = {}, {}, {}
    for feat in FEATURES:
        mu = X_normal[feat].mean()
        sigma = X_normal[feat].std()
        bounds[feat] = (mu - sigma, mu + sigma)              # Eq. 6 / 7
        min_jumps[feat] = 0.30 * sigma                        # Eq. 8
        fault_thresholds[feat] = max(mu - 2.5 * sigma, 5.0)   # Eq. 9

    return bounds, min_jumps, fault_thresholds


def fit_svr_triage(X_train_scaled, y_train):
    """One SVR per sensor channel, trained on normal-only scaled data to
    predict that channel from the other four (Section 3.4.2, Eq. 10)."""
    normal_mask = (y_train == "00").to_numpy()
    X_normal_scaled = X_train_scaled[normal_mask]

    svr_models, residual_thresholds = {}, {}
    for feat in FEATURES:
        X_svr = X_normal_scaled.drop(columns=[feat])
        y_svr = X_normal_scaled[feat]

        svr = SVR(kernel="rbf", C=1.0)
        svr.fit(X_svr, y_svr)
        svr_models[feat] = svr

        residuals = np.abs(y_svr - svr.predict(X_svr))
        residual_thresholds[feat] = np.percentile(residuals, 95)

    return svr_models, residual_thresholds


def svr_categorise(row_scaled, row_orig, bounds, min_jumps, fault_thresholds,
                    svr_models, residual_thresholds):
    """Three-tier Stage 2 triage decision logic (Section 3.4.2)."""
    # Tier 1 — Sensor Fault Check
    if (
        row_orig["Heart_Rate"] < fault_thresholds["Heart_Rate"]
        or row_orig["SpO2"] < fault_thresholds["SpO2"]
    ):
        return "11"

    # Tier 2 — Point Anomaly Check
    for feat in ["Heart_Rate", "SpO2", "Body_Temperature"]:
        actual_val = row_orig[feat]
        mean_val = row_orig[f"{feat}_roll_mean"]
        std_val = row_orig[f"{feat}_roll_std"]

        lo, hi = bounds[feat]
        if actual_val < lo or actual_val > hi:
            return "10"

        if pd.notna(std_val):
            allowed_variance = max(3 * std_val, min_jumps[feat])
            if abs(actual_val - mean_val) > allowed_variance:
                return "10"

    # Tier 3 — Contextual Anomaly Check (SVR residuals)
    context_votes = 0
    for feat in FEATURES:
        input_row = row_scaled.drop(labels=[feat]).to_frame().T
        input_row.columns = [c for c in row_scaled.index if c != feat]
        pred_val = svr_models[feat].predict(input_row)[0]
        residual = abs(row_scaled[feat] - pred_val)
        if residual > residual_thresholds[feat]:
            context_votes += 1

    return "01" if context_votes >= 1 else "10"

In [ ]:
# ==========================================================
# CORE PIPELINE FUNCTIONS — Unified Inference Interface
# ==========================================================
def run_inference(X_scaled_df, X_orig_df, y_true, gatekeeper, models, artifacts):
    """
    Single, consistent interface for all three gatekeepers, sharing the same
    Stage 2 categorisation logic (Section 3.5, Listing 1).

    Parameters
    ----------
    X_scaled_df : pd.DataFrame -- MinMax-scaled feature matrix
    X_orig_df   : pd.DataFrame -- original unscaled features (+ rolling stats)
    y_true      : pd.Series / array -- ground-truth string labels
    gatekeeper  : str -- 'svm' | 'rf' | 'bilstm'
    models      : dict -- fitted gatekeeper models, keyed by name
    artifacts   : dict -- Stage 2 artifacts (bounds, min_jumps,
                  fault_thresholds, svr_models, residual_thresholds)

    Returns
    -------
    pd.DataFrame with columns: Row_ID, Actual, Predicted, Match
    """
    if gatekeeper == "svm":
        gate_preds = models["svm"].predict(X_scaled_df)
        is_abnormal = gate_preds == 1

    elif gatekeeper == "rf":
        rf, le = models["rf"]
        preds = le.inverse_transform(rf.predict(X_scaled_df))
        is_abnormal = preds != "00"

    elif gatekeeper == "bilstm":
        model, le = models["bilstm"]
        X_win = build_windowed_input(X_scaled_df)
        probs = model.predict(X_win, verbose=0)
        preds = le.inverse_transform(np.argmax(probs, axis=1))
        is_abnormal = preds != "00"

    else:
        raise ValueError(f"Unknown gatekeeper: {gatekeeper!r} (expected 'svm', 'rf', or 'bilstm')")

    y_true_arr = np.asarray(y_true)
    rows = []
    for i in range(len(X_scaled_df)):
        if is_abnormal[i]:
            pred_code = svr_categorise(
                X_scaled_df.iloc[i],
                X_orig_df.iloc[i],
                artifacts["bounds"],
                artifacts["min_jumps"],
                artifacts["fault_thresholds"],
                artifacts["svr_models"],
                artifacts["residual_thresholds"],
            )
        else:
            pred_code = "00"

        rows.append(
            {
                "Row_ID": int(X_scaled_df.index[i]),
                "Actual": y_true_arr[i],
                "Predicted": pred_code,
                "Match": y_true_arr[i] == pred_code,
            }
        )

    return pd.DataFrame(rows)


def compute_metrics(results_df):
    """Accuracy / macro-F1 / macro-precision + classification report +
    confusion matrix for a single model's results_df."""
    y_true = results_df["Actual"]
    y_pred = results_df["Predicted"]

    labels = sorted(set(y_true) | set(y_pred))
    target_names = [CLASS_NAMES.get(c, c) for c in labels]

    report_str = classification_report(
        y_true, y_pred, labels=labels, target_names=target_names, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "report_str": report_str,
        "confusion_matrix": cm,
        "labels": labels,
        "target_names": target_names,
    }

## Phase 1 — Data Ingestion & Aggregation
Uploads (or locally loads) all patient telemetry CSVs, concatenates them in numerical order, and derives the four-code ground-truth label from the `Anomaly_Type` and `Normal` columns.

In [ ]:
# ==========================================================
# PHASE 1: DATA INGESTION & AGGREGATION
# ==========================================================
print("=== Phase 1: Data Ingestion ===")

raw_files = load_uploaded_files()
df, sorted_filenames = aggregate_telemetry(raw_files)

print(f"[Info] Aggregated dataset loaded: {len(df)} observations from {len(sorted_filenames)} file(s).\n")
print("[Info] Ground-truth class distribution (full dataset):")
print(df["Actual_Code"].value_counts().rename(index=CLASS_NAMES))

## Phase 2 — Temporal Feature Engineering
Adds 5-step rolling mean/std features per sensor channel (Eq. 1), expanding 5 raw features to 15 (or 17 including any extra columns), and fills the resulting edge NaNs.

In [ ]:
# ==========================================================
# PHASE 2: TEMPORAL FEATURE ENGINEERING
# ==========================================================
print("=== Phase 2: Temporal Feature Extraction ===")

df, X, y = engineer_features(df)

print(f"[Info] Feature engineering complete: {X.shape[1]} features per observation "
      f"(5 raw + 5 rolling means + 5 rolling stds).\n")

## Phase 3 — Train/Test Split & Normalization
Stratified 80/20 split, followed by Min-Max scaling fit only on the training partition (Eq. 2).

In [ ]:
# ==========================================================
# PHASE 3: TRAIN/TEST SPLIT & NORMALIZATION
# ==========================================================
print("=== Phase 3: Data Normalization ===")

X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test, scaler = split_and_scale(X, y)

print(f"[Info] Train: {len(X_train)} samples | Test: {len(X_test)} samples\n")
print("[Info] Train class distribution:")
print(y_train.value_counts().rename(index=CLASS_NAMES))
print("\n[Info] Test class distribution:")
print(y_test.value_counts().rename(index=CLASS_NAMES))

missing_in_test = set(y_train.unique()) - set(y_test.unique())
if missing_in_test:
    print(
        f"\n[Warning] Class(es) {sorted(missing_in_test)} appear in training "
        f"but have zero support in the test split — their precision/recall/F1 "
        f"will report as 0 in the classification report below, and macro "
        f"averages will be pulled down accordingly. This is a property of the "
        f"dataset/split, not a modelling bug — consider whether the source "
        f"telemetry files actually contain Contextual Anomaly (01) and Sensor "
        f"Fault (11) examples."
    )

## Phase 4 — Stage 1: Train All Three Gatekeeper Models
Trains the SVM, Random Forest, and Bi-Directional LSTM gatekeepers described in Sections 3.3.1–3.3.3. Each is trained on a `RandomOverSampler`-balanced version of the training set. The BiLSTM step is skipped gracefully if TensorFlow isn't installed.

In [ ]:
# ==========================================================
# PHASE 4: STAGE 1 — TRAIN ALL THREE GATEKEEPER MODELS
# ==========================================================
print("=== Phase 4: Stage 1 Gatekeeper Training ===")

models = {}
bilstm_history = None

print("[Training] SVM gatekeeper (RBF kernel)...")
models["svm"] = train_svm_gatekeeper(X_train_scaled, y_train)
print("  -> done.")

print("[Training] Random Forest gatekeeper (100 trees, balanced)...")
models["rf"] = train_rf_gatekeeper(X_train_scaled, y_train)
print("  -> done.")

print("[Training] Bi-Directional LSTM gatekeeper (this can take a few minutes)...")
try:
    bilstm_model, bilstm_le, bilstm_history = train_bilstm_gatekeeper(
        X_train_scaled, y_train, epochs=50, patience=5, verbose=1
    )
    models["bilstm"] = (bilstm_model, bilstm_le)
    print("  -> done.")
except ImportError as exc:
    print(f"  -> [Skipped] {exc}")

print(f"\n[Info] Gatekeepers trained: {list(models.keys())}")

## Phase 5 — Stage 2: Dynamic Thresholds & SVR Triage
Computes patient-adaptive dynamic bounds from the normal-only training subset (Eq. 6–9) and fits one SVR per sensor channel for contextual-anomaly residual scoring (Section 3.4).

In [ ]:
# ==========================================================
# PHASE 5: STAGE 2 — DYNAMIC THRESHOLDS & SVR TRIAGE
# ==========================================================
print("=== Phase 5: Stage 2 Triage Setup ===")

bounds, min_jumps, fault_thresholds = fit_dynamic_bounds(X_train, y_train)
svr_models, residual_thresholds = fit_svr_triage(X_train_scaled, y_train)

artifacts = {
    "bounds": bounds,
    "min_jumps": min_jumps,
    "fault_thresholds": fault_thresholds,
    "svr_models": svr_models,
    "residual_thresholds": residual_thresholds,
}

print("[Info] Patient-adaptive dynamic bounds (from normal training subset):")
for feat in FEATURES:
    lo, hi = bounds[feat]
    print(f"  {feat:<18s} bounds=({lo:7.2f}, {hi:7.2f})  "
          f"min_jump={min_jumps[feat]:6.3f}  fault_thresh={fault_thresholds[feat]:7.2f}")

print("\n[Info] Stage 2 SVR triage models fitted for all 5 channels.")

## Phase 6 — Inference & Comparative Evaluation
Runs the full two-stage pipeline for every trained gatekeeper via the unified `run_inference()` interface, and builds the comparative performance table (Table 5 in the report).

In [ ]:
# ==========================================================
# PHASE 6: INFERENCE & COMPARATIVE EVALUATION
# ==========================================================
print("=== Phase 6: Running Full Two-Stage Inference for Each Gatekeeper ===")

MODEL_LABELS = {"svm": "Hybrid SVM-SVR", "rf": "Random Forest", "bilstm": "Bi-Directional LSTM"}

results = {}
metrics = {}

for name in models:
    print(f"\n[Inference] Gatekeeper = {MODEL_LABELS[name]}")
    results_df = run_inference(X_test_scaled, X_test, y_test, name, models, artifacts)
    results[name] = results_df

    m = compute_metrics(results_df)
    metrics[name] = m

    print(f"  Accuracy: {m['accuracy']*100:.2f}%  |  Macro F1: {m['f1_macro']:.4f}  |  "
          f"Macro Precision: {m['precision_macro']:.4f}")
    print(m["report_str"])

# ---- Comparison table ----
comparison_df = pd.DataFrame(
    [
        {
            "Model": MODEL_LABELS[name],
            "Accuracy (%)": round(metrics[name]["accuracy"] * 100, 2),
            "F1 (Macro)": round(metrics[name]["f1_macro"], 4),
            "Precision (Macro)": round(metrics[name]["precision_macro"], 4),
        }
        for name in models
    ]
).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)

print("\n=== Comparative Model Performance ===")
print(comparison_df.to_string(index=False))

## Phase 7 — Visualizations
Confusion matrices, class distributions, comparative accuracy/F1 charts, BiLSTM training curves, and Random Forest feature importances — matching Figures 4–9 of the report.

In [ ]:
# ==========================================================
# PHASE 7a: CONFUSION MATRIX & CLASS DISTRIBUTION (per model)
# ==========================================================
print("=== Phase 7a: Confusion Matrices & Class Distributions ===")

os.makedirs("outputs", exist_ok=True)

for name in models:
    m = metrics[name]
    results_df = results[name]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    sns.heatmap(
        m["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
        xticklabels=m["labels"], yticklabels=m["labels"], ax=axes[0], cbar=True,
    )
    axes[0].set_title(f"Confusion Matrix — {MODEL_LABELS[name]}")
    axes[0].set_xlabel("Predicted Label")
    axes[0].set_ylabel("True Label")

    dist_df = pd.DataFrame(
        {
            "Actual": results_df["Actual"].value_counts(),
            "Predicted": results_df["Predicted"].value_counts(),
        }
    ).fillna(0).reindex(sorted(set(results_df["Actual"]) | set(results_df["Predicted"])))
    dist_df.plot(kind="bar", ax=axes[1], color=["#3B82F6", "#F59E0B"])
    axes[1].set_title(f"Class Distribution — {MODEL_LABELS[name]}")
    axes[1].set_xlabel("Anomaly Code")
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis="x", rotation=0)

    plt.tight_layout()
    fig_path = f"outputs/confusion_matrix_{name}.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print(f"[Saved] {fig_path}")

In [ ]:
# ==========================================================
# PHASE 7b: COMPARATIVE PERFORMANCE CHARTS
# ==========================================================
print("=== Phase 7b: Comparative Performance Charts ===")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

model_names_plot = [MODEL_LABELS[n] for n in models]
accuracies = [metrics[n]["accuracy"] * 100 for n in models]
f1_scores = [metrics[n]["f1_macro"] for n in models]
precisions = [metrics[n]["precision_macro"] for n in models]

bars = axes[0].bar(model_names_plot, accuracies, color=["#3B82F6", "#10B981", "#F97316"][: len(models)])
axes[0].set_title("Detection Accuracy — All Models")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 1, f"{val:.2f}%", ha="center")

x = np.arange(len(models))
width = 0.35
axes[1].bar(x - width / 2, f1_scores, width, label="F1 (macro)", color="#3B82F6")
axes[1].bar(x + width / 2, precisions, width, label="Precision (macro)", color="#93C5FD")
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names_plot)
axes[1].set_title("Macro F1-Score & Precision — All Models")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.tight_layout()
plt.savefig("outputs/comparison_chart.png", dpi=150)
plt.show()
print("[Saved] outputs/comparison_chart.png")

# NOTE: The report's Table 5 also lists a "Static Threshold (baseline)" row
# (~62% accuracy). That baseline is NOT computed anywhere in this notebook,
# so it is intentionally left out of this chart rather than hard-coding an
# unverified number. Implement a simple fixed-threshold classifier here if
# you want an apples-to-apples baseline comparison.

In [ ]:
# ==========================================================
# PHASE 7c: BiLSTM TRAINING HISTORY
# ==========================================================
if bilstm_history is not None:
    print("=== Phase 7c: BiLSTM Training History ===")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(bilstm_history.history["accuracy"], label="Train")
    axes[0].plot(bilstm_history.history["val_accuracy"], label="Val", linestyle="--")
    axes[0].set_title("BiLSTM — Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(bilstm_history.history["loss"], label="Train")
    axes[1].plot(bilstm_history.history["val_loss"], label="Val", linestyle="--")
    axes[1].set_title("BiLSTM — Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("outputs/bilstm_training_history.png", dpi=150)
    plt.show()
    print("[Saved] outputs/bilstm_training_history.png")
else:
    print("[Skipped] BiLSTM was not trained (TensorFlow unavailable) — no training history to plot.")

In [ ]:
# ==========================================================
# PHASE 7d: RANDOM FOREST FEATURE IMPORTANCES
# ==========================================================
if "rf" in models:
    print("=== Phase 7d: Random Forest Feature Importances ===")

    rf_model, _ = models["rf"]
    importances = pd.Series(rf_model.feature_importances_, index=X_train_scaled.columns)
    importances = importances.sort_values(ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=(9, 6))
    importances.plot(kind="barh", ax=ax, color="#3B82F6")
    ax.set_title("Random Forest — Top Feature Importances")
    ax.set_xlabel("Importance")

    plt.tight_layout()
    plt.savefig("outputs/rf_feature_importance.png", dpi=150)
    plt.show()
    print("[Saved] outputs/rf_feature_importance.png")
else:
    print("[Skipped] Random Forest gatekeeper was not trained.")

## Phase 8 — Export Results
Saves per-model prediction CSVs and the comparison table to `./outputs/` for use in the report or the Streamlit dashboard.

In [ ]:
# ==========================================================
# PHASE 8: EXPORT PREDICTIONS & SUMMARY
# ==========================================================
print("=== Phase 8: Exporting Results ===")

os.makedirs("outputs", exist_ok=True)

for name, results_df in results.items():
    out_path = f"outputs/predictions_{name}.csv"
    results_df.to_csv(out_path, index=False)
    print(f"[Saved] {out_path}")

comparison_df.to_csv("outputs/model_comparison.csv", index=False)
print("[Saved] outputs/model_comparison.csv")

best_model = comparison_df.iloc[0]
print(
    f"\n[Summary] Best performing gatekeeper: {best_model['Model']} "
    f"({best_model['Accuracy (%)']:.2f}% accuracy, "
    f"F1={best_model['F1 (Macro)']:.4f}, "
    f"Precision={best_model['Precision (Macro)']:.4f})"
)